# Notebook 04 — Dashboard Gradio (Demo cuối)

Load các artifact đã lưu từ `03_hybrid_ensemble.ipynb` và bật web UI bằng **Gradio**.

**Yêu cầu trước:** đã chạy `03_hybrid_ensemble.ipynb` để có:
- `models/svm_pipeline.pkl`
- `models/bilstm.pt` + `models/bilstm_vocab.json`
- `models/phobert_head.pkl` *(nếu môi trường có)*
- `models/hybrid_meta.json`

**Dashboard cho phép:**
1. Nhập **một câu** tiếng Việt vào ô input.
2. Xem **nhãn cảm xúc cuối** + **xác suất 3 lớp** từ Hybrid Ensemble.
3. Xem **xác suất từng component** (SVM / BiLSTM / PhoBERT) — để biết model nào đồng ý / phản đối.
4. **Cơ sở phân tích (Explainability):** highlight các **từ ủng hộ** (tô xanh) và **từ phản đối** (tô đỏ) nhãn dự đoán, kèm bảng top từ ảnh hưởng nhất. Dùng coefficient của SVM × giá trị TF-IDF — đây là cách giải thích **chính xác về toán** (không phải LIME/SHAP xấp xỉ).

In [1]:
# --- Imports & paths --------------------------------------------------------
import json, re, string
from pathlib import Path
import numpy as np
import joblib
import torch
import torch.nn as nn

ROOT = Path('..').resolve()
MODELS = ROOT / 'models'

# --- Load meta (xác định component nào có sẵn) ------------------------------
with open(MODELS / 'hybrid_meta.json', 'r', encoding='utf-8') as f:
    META = json.load(f)
LABEL_NAMES = {int(k): v for k, v in META['label_names'].items()}
NUM_CLASSES = len(LABEL_NAMES)
COMPONENTS = META['components']     # list các tên đang dùng
WEIGHTS = np.array(META['weights']) # đã chuẩn hoá tổng 1
print('Components:', COMPONENTS)
print('Weights   :', dict(zip(COMPONENTS, [round(w, 4) for w in WEIGHTS])))

Components: ['SVM', 'BiLSTM', 'PhoBERT']
Weights   : {'SVM': np.float64(0.3333), 'BiLSTM': np.float64(0.3333), 'PhoBERT': np.float64(0.3333)}


In [2]:
# --- Preprocess (giống notebook 00 / 03, copy gọn để self-contained) -------
from underthesea import word_tokenize as _wt
_PUNCT = string.punctuation + '“”‘’–—…•·«»\u200b'
_URL_RE = re.compile(r'http\S+|www\.\S+')
_DIGIT_RE = re.compile(r'\d+')
_SPACE_RE = re.compile(r'\s+')
_EMOJI_RE = re.compile('[\U0001F300-\U0001F9FF\U0001FA00-\U0001FAFF\u2600-\u27BF]+')
_VN_STOP = set([
    'và','là','của','có','được','trong','cho','khi','thì','đã','đang','sẽ','cũng',
    'này','đó','kia','để','như','với','mà','một','các','những','rất','quá','lắm',
    'ạ','nhé','nha','ơi','thế','vậy','rồi','nữa','lại','chỉ','vẫn','còn','hơn','bị',
    'từ','về','trên','dưới','ngoài','bên','theo','vào','ra','lên','xuống',
])

def preprocess(text: str) -> str:
    t = str(text).lower()
    t = _URL_RE.sub(' ', t); t = _EMOJI_RE.sub(' ', t); t = _DIGIT_RE.sub(' ', t)
    t = t.translate(str.maketrans('', '', _PUNCT))
    t = _SPACE_RE.sub(' ', t).strip()
    t = _wt(t, format='text')
    return ' '.join(w for w in t.split() if w not in _VN_STOP)

In [ ]:
# --- Load SVM pipeline (đã đóng gói TF-IDF + classifier) -------------------
svm_pipe = joblib.load(MODELS / 'svm_pipeline.pkl')

# --- Load BiLSTM (weights + vocab) -----------------------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with open(MODELS / 'bilstm_vocab.json', 'r', encoding='utf-8') as f:
    BIL = json.load(f)
WORD2IDX = BIL['word2idx']; MAX_LEN = BIL['max_len']

class BiLSTMClassifier(nn.Module):
    """Phải khớp y nguyên kiến trúc đã train trong notebook 03 — nếu không khớp,
    load_state_dict sẽ fail. Cho phép đọc cấu hình từ json để dễ chỉnh."""
    def __init__(self, vocab_size, embed_dim, hidden, n_classes, dropout):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, n_classes)
    def forward(self, x):
        mask = (x != 0).unsqueeze(-1).float()
        h = self.emb(x); out, _ = self.lstm(h)
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.fc(self.dropout(pooled))

bilstm = BiLSTMClassifier(
    vocab_size=len(WORD2IDX),
    embed_dim=BIL['embed_dim'],
    hidden=BIL['hidden'],
    n_classes=BIL['num_classes'],
    dropout=BIL['dropout'],
).to(DEVICE)
bilstm.load_state_dict(torch.load(MODELS / 'bilstm.pt', map_location=DEVICE))
bilstm.eval()

def _encode(text):
    ids = [WORD2IDX.get(w, WORD2IDX['<unk>']) for w in text.split()][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

# --- Load PhoBERT head nếu có ----------------------------------------------
phobert_tok = phobert_model = bert_head = None
if 'PhoBERT' in COMPONENTS and META.get('phobert_model_name'):
    try:
        from transformers import AutoTokenizer, AutoModel
        phobert_tok = AutoTokenizer.from_pretrained(META['phobert_model_name'])
        phobert_model = AutoModel.from_pretrained(META['phobert_model_name']).to(DEVICE).eval()
        bert_head = joblib.load(MODELS / 'phobert_head.pkl')
        print('[OK] PhoBERT component active')
    except Exception as e:
        print('[WARN] PhoBERT bị bỏ:', e)
print('All components loaded.')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[OK] PhoBERT component active
All components loaded.


In [4]:
# --- Explainability dùng coefficient SVM × TF-IDF --------------------------
# Logic: SVM tuyến tính, score lớp c cho câu x  =  Σ (coef[c, i] * tfidf[i])
#   → đóng góp của token i vào lớp c = coef[c, i] * tfidf[i]
# Đây là **giải thích chính xác** (không xấp xỉ như LIME/SHAP) cho SVM nhánh.
# Lấy trung bình coef qua các fold của CalibratedClassifierCV cho ổn định.
_svm_vec = svm_pipe.named_steps['tfidf']
_svm_clf = svm_pipe.named_steps['clf']
_svm_coefs = np.mean(
    [cc.estimator.coef_ for cc in _svm_clf.calibrated_classifiers_],
    axis=0,
)  # shape: (num_classes, num_features)
_svm_features = np.array(_svm_vec.get_feature_names_out())

def explain_svm(text_clean: str, target_class: int, top_k: int = 5):
    """Trả về (top_support, top_against) cho lớp `target_class`.
    
    - top_support: [(token, score)] — token có score > 0, kéo câu về phía target_class
    - top_against: [(token, score)] — token có score < 0, kéo ngược lại
    
    Vì TF-IDF dùng (1,2)-gram, "token" có thể là 1 hoặc 2 từ.
    """
    X = _svm_vec.transform([text_clean])
    nz = X.nonzero()[1]
    if len(nz) == 0:
        return [], []
    coefs = _svm_coefs[target_class]  # vector trên feature space
    contrib = [(_svm_features[i], float(coefs[i] * X[0, i])) for i in nz]
    pos = sorted([c for c in contrib if c[1] > 0], key=lambda x: -x[1])[:top_k]
    neg = sorted([c for c in contrib if c[1] < 0], key=lambda x: x[1])[:top_k]
    return pos, neg

def highlight_sentence(text_clean: str, target_class: int) -> str:
    """Tô màu nền cho từng token theo dấu của contribution. Trả về HTML.
    
    Xanh: ủng hộ nhãn; Đỏ: phản đối nhãn; cường độ màu tỉ lệ |contribution|.
    """
    X = _svm_vec.transform([text_clean])
    nz = X.nonzero()[1]
    # Map feature_name -> contribution
    contrib_map = {_svm_features[i]: float(_svm_coefs[target_class, i] * X[0, i]) for i in nz}
    # Tìm max abs để normalize cường độ màu
    max_abs = max((abs(v) for v in contrib_map.values()), default=1.0) or 1.0

    spans = []
    for w in text_clean.split():
        # Tìm match unigram (TF-IDF 1-gram lưu trực tiếp token).
        # Cụm 2-gram không tô riêng vì khó render trên HTML đơn giản — chấp nhận trade-off.
        c = contrib_map.get(w, 0.0)
        if c > 0:
            alpha = min(0.6, abs(c) / max_abs * 0.6 + 0.15)
            spans.append(f'<span style="background:rgba(46,204,113,{alpha});'
                         f'padding:2px 5px;margin:2px;border-radius:4px" '
                         f'title="+{c:.3f}">{w}</span>')
        elif c < 0:
            alpha = min(0.6, abs(c) / max_abs * 0.6 + 0.15)
            spans.append(f'<span style="background:rgba(231,76,60,{alpha});'
                         f'padding:2px 5px;margin:2px;border-radius:4px" '
                         f'title="{c:.3f}">{w}</span>')
        else:
            spans.append(f'<span style="padding:2px 5px;margin:2px">{w}</span>')
    return ('<div style="line-height:2.2;font-size:16px">' +
            ' '.join(spans) + '</div>')

# --- Hàm tổng hợp: predict + explain ---------------------------------------
def analyze(text: str):
    """Hàm chính cho Gradio. Trả về 5 output:
      1. label_text   — string nhãn cuối + %
      2. final_probs  — dict {label: prob} cho widget gr.Label
      3. per_comp     — dict {component: {label: prob}} cho gr.JSON
      4. explain_html — câu input có highlight theo độ đóng góp
      5. top_table    — DataFrame hiển thị top từ ủng hộ/phản đối
    """
    if not text or not text.strip():
        return ('Vui lòng nhập câu.', {}, {}, '', None)

    clean = preprocess(text)
    probs_list = []

    # SVM
    p_svm = svm_pipe.predict_proba([clean])[0]
    probs_list.append(p_svm)

    # BiLSTM
    ids = torch.tensor([_encode(clean)], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        p_lstm = torch.softmax(bilstm(ids), dim=1).cpu().numpy()[0]
    probs_list.append(p_lstm)

    # PhoBERT (nếu active)
    if 'PhoBERT' in COMPONENTS and phobert_model is not None:
        with torch.no_grad():
            enc = phobert_tok([clean], padding=True, truncation=True,
                              max_length=128, return_tensors='pt').to(DEVICE)
            feat = phobert_model(**enc).last_hidden_state[:, 0, :].cpu().numpy()
        p_bert = bert_head.predict_proba(feat)[0]
        probs_list.append(p_bert)

    stacked = np.stack(probs_list, axis=0)
    final = (stacked * WEIGHTS[:len(probs_list), None]).sum(axis=0)
    final = final / final.sum()
    label = int(final.argmax())

    final_dict = {LABEL_NAMES[i]: float(final[i]) for i in range(NUM_CLASSES)}
    per_comp = {
        n: {LABEL_NAMES[i]: round(float(p[i]), 4) for i in range(NUM_CLASSES)}
        for n, p in zip(COMPONENTS, probs_list)
    }
    label_text = f'{LABEL_NAMES[label]} ({final[label]*100:.1f}%)'

    # Explainability cho nhãn dự đoán
    pos, neg = explain_svm(clean, label, top_k=5)
    explain_html = (
        f'<p><b>Câu đã tiền xử lý:</b> <code>{clean}</code></p>'
        f'<p><b>Highlight đóng góp vào nhãn "{LABEL_NAMES[label]}"</b> '
        f'(xanh = ủng hộ, đỏ = phản đối):</p>'
        + highlight_sentence(clean, label)
    )

    # Bảng top từ ảnh hưởng (dạng list-of-list để Gradio render Dataframe)
    rows = []
    for w, s in pos:
        rows.append([w, f'+{s:.3f}', 'Ủng hộ'])
    for w, s in neg:
        rows.append([w, f'{s:.3f}', 'Phản đối'])
    if not rows:
        rows = [['(không có token đủ mạnh)', '0.000', '-']]
    import pandas as _pd
    top_df = _pd.DataFrame(rows, columns=['Token', 'Đóng góp', 'Hướng'])

    return label_text, final_dict, per_comp, explain_html, top_df

# Test nhanh
_r = analyze('Sản phẩm rất tốt, giao hàng nhanh')
print('Label:', _r[0])
print('Final:', _r[1])

Label: Positive (35.2%)
Final: {'Negative': 0.3509817017852783, 'Neutral': 0.2973010906822118, 'Positive': 0.35171720753250985}


In [ ]:
# --- DL baseline comparison (from notebook 02) -------------------------------
import pandas as pd
import matplotlib.pyplot as plt

DL_RESULTS_PATH = ROOT / 'results' / 'dl_baseline.csv'
DL_CURVES_PATH = ROOT / 'results' / 'dl_curves.png'
DL_CM_PATH = ROOT / 'results' / 'dl_confusion_matrix.png'

def load_dl_results():
    if DL_RESULTS_PATH.exists():
        return pd.read_csv(DL_RESULTS_PATH)
    return pd.DataFrame(columns=['model', 'accuracy', 'precision', 'recall', 'f1_macro'])

def plot_dl_metrics(df):
    if df.empty:
        return None
    metrics = ['accuracy', 'precision', 'recall', 'f1_macro']
    df_plot = df.set_index('model')[metrics]
    ax = df_plot.plot(kind='bar', figsize=(6, 3), ylim=(0, 1), title='DL Baseline Metrics')
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)
    fig = ax.get_figure()
    fig.tight_layout()
    return fig

def load_dl_dashboard():
    df = load_dl_results()
    fig = plot_dl_metrics(df)
    curves = str(DL_CURVES_PATH) if DL_CURVES_PATH.exists() else None
    cm = str(DL_CM_PATH) if DL_CM_PATH.exists() else None
    return df, fig, curves, cm

DL_DF, DL_FIG, DL_CURVES, DL_CM = load_dl_dashboard()

## Gradio UI

Chạy cell dưới → mở link local. Dashboard có 2 tab:
- **Hybrid Ensemble**: demo dự đoán + explainability.
- **DL Baselines**: so sánh LSTM/BiLSTM/GRU từ notebook 02.

In [ ]:
import gradio as gr

EXAMPLES = [
    'Sản phẩm rất tốt, giao hàng nhanh',
    'Hàng kém chất lượng, không như mô tả',
    'Bài giảng buồn ngủ, nội dung lặp lại nhiều',
    'Giảng viên giảng rất dễ hiểu và nhiệt tình',
    'Lớp học diễn ra đúng giờ quy định',
]

# UI structure: 2 tab — Hybrid + DL Baselines
with gr.Blocks(title='Vietnamese Sentiment — Hybrid Ensemble',
               theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Phân tích cảm xúc tiếng Việt — Dashboard tổng hợp')

    with gr.Tabs():
        # ---------------- Tab 1: Hybrid Ensemble ----------------------------
        with gr.Tab('Hybrid Ensemble'):
            gr.Markdown(
                f'**Components active:** {", ".join(COMPONENTS)}  |  '
                f'**Trọng số voting:** '
                + ', '.join(f'`{n}`={w:.2f}' for n, w in zip(COMPONENTS, WEIGHTS))
            )

            with gr.Row():
                # --- Cột trái: input + nhãn cuối ---------------------------
                with gr.Column(scale=1):
                    inp = gr.Textbox(
                        label='Nhập một câu tiếng Việt',
                        lines=3,
                        placeholder='Ví dụ: Sản phẩm rất tốt, giao hàng nhanh',
                    )
                    btn = gr.Button('Phân tích cảm xúc', variant='primary', size='lg')
                    out_label = gr.Textbox(label='Kết quả cuối', interactive=False)
                    out_final = gr.Label(
                        label='Xác suất (Hybrid Ensemble)',
                        num_top_classes=3,
                    )
                    gr.Examples(EXAMPLES, inp, label='Câu mẫu (bấm để thử)')

                # --- Cột phải: explainability + per-component --------------
                with gr.Column(scale=1):
                    gr.Markdown('### Cơ sở phân tích (Explainability)')
                    out_html = gr.HTML(label='Highlight từ')
                    out_top = gr.Dataframe(
                        headers=['Token', 'Đóng góp', 'Hướng'],
                        label='Top 5 từ ảnh hưởng nhất',
                        interactive=False, wrap=True,
                    )
                    gr.Markdown('### Đóng góp của từng model trong Hybrid')
                    out_per = gr.JSON(label='Xác suất theo component')

            gr.Markdown(
                '**Ghi chú giải thích:** chỉ số đóng góp = `coefficient_SVM × TF-IDF_giá_trị`. '
                'Số dương (xanh) = từ kéo câu về nhãn dự đoán; '
                'Số âm (đỏ) = từ phản đối nhãn này (kéo về nhãn khác). '
                'Đây là cách giải thích **chính xác về toán** cho nhánh SVM, không phải xấp xỉ.'
            )

            btn.click(
                analyze, inputs=inp,
                outputs=[out_label, out_final, out_per, out_html, out_top],
            )
            # Cho phép nhấn Enter trong textbox = trigger analyze
            inp.submit(
                analyze, inputs=inp,
                outputs=[out_label, out_final, out_per, out_html, out_top],
            )

        # ---------------- Tab 2: DL Baselines -----------------------------
        with gr.Tab('DL Baselines'):
            gr.Markdown(
                'So sánh LSTM/BiLSTM/GRU từ `02_dl_baseline.ipynb`. '
                'Nếu chưa có file kết quả, hãy chạy notebook 02 trước.'
            )
            dl_refresh = gr.Button('Reload kết quả DL', variant='secondary')
            dl_table = gr.Dataframe(
                value=DL_DF, label='Bảng so sánh metrics',
                interactive=False, wrap=True,
            )
            dl_plot = gr.Plot(value=DL_FIG, label='So sánh Accuracy/Precision/Recall/F1')
            with gr.Row():
                dl_curves = gr.Image(value=DL_CURVES, label='Training curves (dev)')
                dl_cm = gr.Image(value=DL_CM, label='Confusion matrix (best model)')

            dl_refresh.click(
                load_dl_dashboard,
                outputs=[dl_table, dl_plot, dl_curves, dl_cm],
            )

# share=False để chạy local; đổi share=True nếu cần link tạm public.
# inbrowser=True tự mở browser mặc định khi launch.
demo.launch(share=False, inbrowser=True)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_2276\215153067.py:12: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title='Vietnamese Sentiment — Hybrid Ensemble',


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
